# Arm Simulation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import mujoco
import numpy as np

from src.model.arm import Arm
from src.model.solver import OnlineSolver
from src.sim.trajectory import Trajectory
from src.sim.simulation import Simulation, pid_grid_search
from src.sim.simulation_io import (experiment_path, save_simulations, summary_table,
                                   plot_angles, plot_torques, plot_activations, plot_hand_path)

print(f"MuJoCo {mujoco.__version__}")

## Trajectory Generation

In [ ]:
TRAJECTORY = "minjerk" # "minjerk", "cartesian", "step"

PATH_MUJOCO_MODEL = "assets/myo_sim/arm/myoarm.xml"
RESOLUTION_MS = 1.0 # [ms] MuJoCo timestep
TIME_PREP_MS = 100.0

# [shoulder, elbow]
THETA_START = [-20.0, 20.0]
THETA_END = [70.0, 90.0]

# for "step"
STEP_DURATION_MS = 500.0

# for "minjerk" and "cartesian"
TIME_MOVE_MS = 500.0
TIME_LOCKED_MS = 150.0
TIME_POST_MS = 350.0

arm = Arm.planar(PATH_MUJOCO_MODEL)
if TRAJECTORY == "step":
    traj = Trajectory.step(THETA_START, THETA_END, duration_ms=STEP_DURATION_MS,
                           resolution_ms=RESOLUTION_MS, time_prep_ms=TIME_PREP_MS, arm=arm)
elif TRAJECTORY == "minjerk":
    traj = Trajectory.minjerk(THETA_START, THETA_END, resolution_ms=RESOLUTION_MS,
                              time_prep_ms=TIME_PREP_MS, time_move_ms=TIME_MOVE_MS,
                              time_locked_ms=TIME_LOCKED_MS, time_post_ms=TIME_POST_MS,
                              hold_in_post=True, arm=arm)
elif TRAJECTORY == "cartesian":
    traj = Trajectory.cartesian(arm, THETA_START, THETA_END, resolution_ms=RESOLUTION_MS,
                                time_prep_ms=TIME_PREP_MS, time_move_ms=TIME_MOVE_MS,
                                time_locked_ms=TIME_LOCKED_MS, time_post_ms=TIME_POST_MS)
else:
    raise ValueError(f"TRAJECTORY: {TRAJECTORY} not valid.")
print(traj)
traj.plot(show_phases=True)
traj.plot_hand()
traj.plot_views(["top", "front"])

## Parameters

In [ ]:
W_ref = 0.5
P_VALUE = 3

SIM_ROOT = "simulations/arm"

TIMESTEP_MS = traj.resolution_ms  # [ms] MuJoCo timestep = resolution of the trajectory

## Upload MuJoCo model and solver

In [ ]:
model = arm.simulation_model(timestep_ms=TIMESTEP_MS)
print(arm)

allocators = {"solver": OnlineSolver(arm, p_value=P_VALUE)}

## Workspace flatness

In [ ]:
z = np.array([arm.hand_position(theta)[2] for theta in arm.workspace_grid(9)])
print(f"hand height on the workspace: {z.min():.3f} .. {z.max():.3f} m (excursion {np.ptp(z) * 100:.1f} cm)")

## Simulation

In [ ]:
# one PID per DoF [shoulder, elbow]
K_P = [4.0, 2.0] # [Nm/deg]
K_I = [2.0, 1.0] # [Nm/(deg s)]
K_D = [0.1, 0.05] # [Nm s/deg]

DELAY_MS = 0.0
FEEDFORWARD = True
SETTLED_START = True
SAVE = True

use_ff = FEEDFORWARD and traj.kind != "step"
sims = {}
for name, allocator in allocators.items():
    print(f"Simulating: {name}")
    sims[name] = Simulation.run(model, arm, allocator, W_ref, traj, K_P=K_P, K_I=K_I, K_D=K_D,
                                delay_ms=DELAY_MS, feedforward=use_ff, settled_start=SETTLED_START, label=name)
save_path = experiment_path(sims, SIM_ROOT)

if SAVE:
    save_simulations(sims, save_path)
summary_table(sims).T.round(3)

## Plots

In [ ]:
SAVE_PLOTS = False
save_dir = save_path.replace("simulations", "plots", 1) if SAVE_PLOTS else None

plot_angles(sims, save_dir=save_dir)
plot_torques(sims, save_dir=save_dir)
plot_activations(sims, save_dir=save_dir)
plot_activations(sims, state=True, save_dir=save_dir)
plot_hand_path(sims, save_dir=save_dir)

## PID Tuning

In [ ]:
# PID_ALLOCATOR = "solver"

# results = pid_grid_search(model, arm, allocators[PID_ALLOCATOR], W_ref, traj,
#                           K_P_values=[[2.0, 1.0], [4.0, 2.0]],
#                           K_I_values=[[1.0, 0.5], [2.0, 1.0]],
#                           K_D_values=[[0.1, 0.05]],
#                           delay_ms=DELAY_MS, feedforward=use_ff, settled_start=SETTLED_START)
# results.head(10)